# 1. start with small idea of RAG - search by keywords matching

In [34]:
import os
import glob
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import functools
from concurrent.futures import ThreadPoolExecutor
import time

In [35]:
# LangChain imports
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document, BaseRetriever
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.retrievers import EnsembleRetriever
from langchain_core.callbacks import StdOutCallbackHandler
from typing import List, Dict

In [36]:
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

In [37]:
# Load environment variables
load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [38]:
# Configuration
MODEL = "gpt-4o-mini"
db_name = "medical_vector_db"

In [39]:
class MedicalRAGSystem:
    def __init__(self, use_openai_embeddings=True):
        self.use_openai_embeddings = use_openai_embeddings
        self.context = {}
        self.chunks = []
        self.vectorstore = None
        self.conversation_chain = None
        self.system_message = self._create_system_message()
        
    def _create_system_message(self):
        return (
            "Bạn là một trợ lý y tế AI chuyên nghiệp tại phòng khám. "
            "Nhiệm vụ của bạn là:\n"
            "1. Hỗ trợ dự đoán bệnh dựa trên triệu chứng\n"
            "2. Chỉ định khoa khám phù hợp\n"
            "3. Đưa ra khuyến cáo về sức khỏe\n"
            "4. Cung cấp thông tin về bác sĩ và dịch vụ\n"
            "5. Tư vấn về các vấn đề sức khỏe tổng quát\n\n"
            "LƯU Ý QUAN TRỌNG:\n"
            "- Bạn KHÔNG thay thế cho bác sĩ thật\n"
            "- Luôn khuyến khích bệnh nhân đến gặp bác sĩ để chẩn đoán chính xác\n"
            "- Không đưa ra chẩn đoán cuối cùng\n"
            "- Nếu không chắc chắn, hãy nói rõ ràng\n"
            "- Ưu tiên an toàn bệnh nhân\n"
            "Trả lời bằng tiếng Việt một cách chuyên nghiệp và dễ hiểu."
        )

 

In [40]:
@functools.lru_cache(maxsize=None)
def load_medical_context(self):
        """Load medical knowledge base"""
        context = {}
        
        # Load different types of medical documents
        def load_files_from_folder(folder_path, description):
            files_context = {}
            if os.path.exists(folder_path):
                files = glob.glob(f"{folder_path}/*.md") + glob.glob(f"{folder_path}/*.txt")
                
                def load_single_file(file_path):
                    filename = os.path.basename(file_path).replace('.md', '').replace('.txt', '')
                    try:
                        with open(file_path, "r", encoding="utf-8") as f:
                            return filename, f.read()
                    except Exception as e:
                        print(f"Error loading {file_path}: {e}")
                        return filename, ""
                
                with ThreadPoolExecutor(max_workers=4) as executor:
                    results = executor.map(load_single_file, files)
                    files_context.update(dict(results))
            
            return files_context

In [41]:
# Giả sử bạn đã có hàm load_files_from_folder
def load_files_from_folder(folder, desc):
    """Đọc file trong folder và trả về dict {desc: [tên file]}"""
    import os, glob
    data = {}
    files = glob.glob(os.path.join(folder, "*.md"))
    data[desc] = [os.path.basename(f) for f in files]
    return data


class MedicalBot:
    def load_medical_context(self):
        print("Loading medical context...")

        medical_folders = [
            ("medical-knowledge/diseases", "Thông tin bệnh lý"),
            ("medical-knowledge/symptoms", "Triệu chứng"),
            ("medical-knowledge/departments", "Khoa khám"),
            ("medical-knowledge/doctors", "Thông tin bác sĩ"),
            ("medical-knowledge/treatments", "Điều trị"),
            ("medical-knowledge/health-tips", "Lời khuyên sức khỏe"),
            ("medical-knowledge/services", "Dịch vụ phòng khám")
        ]

        context = {}
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=4) as executor:
            futures = [executor.submit(load_files_from_folder, folder, desc) 
                       for folder, desc in medical_folders]
            
            for future in futures:
                context.update(future.result())

        return context


# 🔹 Test chạy
bot = MedicalBot()
context = bot.load_medical_context()

print("Dữ liệu nạp được:")
print(context)


Loading medical context...
Dữ liệu nạp được:
{'Thông tin bệnh lý': ['benh-vay-nen.md', 'cam_cum.md', 'cao-huyet-ap.md', 'covid-19.md', 'dai-thao-duong.md', 'dot-quy.md', 'hen-phe-quan.md', 'hoi-chung-bau-cuu.md', 'lao-phoi.md', 'loang-xuong.md', 'nam-da.md', 'roi-loan-lo-au.md', 'roi-loan-pho-tu-ky.md', 'soi-than.md', 'soi.md', 'sot-xuat-huyet.md', 'suy-than-man.md', 'suy-tim.md', 'tay-chan-mieng.md', 'thoai-hoa-khop-goi.md', 'tien-san-giat.md', 'tieu_chay.md', 'tram-cam.md', 'ung-thu-da-day.md', 'ung-thu-gan.md', 'viem-da-co-dia.md', 'viem-da-day-ta-trang.md', 'viem-gan-b.md', 'viem-khop-dang-thap.md', 'viem-phoi.md', 'viem_hong.md', 'xo-gan.md'], 'Triệu chứng': ['cam_cum.md', 'ho.md', 'kho-tho.md', 'sot.md'], 'Khoa khám': ['khoa-cap-cuu.md', 'khoa-chan-doan-hinh-anh.md', 'khoa-da-lieu.md', 'khoa-ngoai.md', 'khoa-nhi.md', 'khoa-noi.md', 'khoa-san.md', 'trung-tam-tiem-chung.md'], 'Thông tin bác sĩ': ['dang_quoc_tien.md', 'do_thi_hong.md', 'le_thanh_binh.md', 'le_thi_mai.md', 'le_van_hu

In [42]:
# Create medical knowledge folders if they don't exist
medical_folders = [
            "medical-knowledge/diseases",
            "medical-knowledge/symptoms", 
            "medical-knowledge/departments",
            "medical-knowledge/doctors",
            "medical-knowledge/treatments",
            "medical-knowledge/health-tips",
            "medical-knowledge/services"
        ]
        
for folder in medical_folders:
            os.makedirs(folder, exist_ok=True)

In [43]:
# Load existing documents
folders = glob.glob("medical-knowledge/*")
text_loader_kwargs = {'autodetect_encoding': True}

documents = []
for folder in folders:
    if os.path.isdir(folder):
        doc_type = os.path.basename(folder)
        loader = DirectoryLoader(
            folder, 
            glob="**/*.md", 
            loader_cls=TextLoader, 
            loader_kwargs=text_loader_kwargs
        )
        try:
            folder_docs = loader.load()
            for doc in folder_docs:
                doc.metadata["doc_type"] = doc_type
                documents.append(doc)
        except Exception as e:
            print(f"Error loading from {folder}: {e}")


In [44]:
import os
import glob
import gradio as gr
from typing import List
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.retrievers import EnsembleRetriever
from langchain.schema import BaseRetriever
from pydantic import Field

MODEL = "gpt-3.5-turbo"
DB_NAME = "medical_db"


# =========================
# Custom Keyword Retriever
# =========================
class MedicalKeywordRetriever(BaseRetriever):
    """Custom keyword retriever for medical terms"""

    context_dict: dict = Field(default_factory=dict)

    def _get_relevant_documents(self, query: str) -> List[Document]:
        relevant_docs = []
        query_lower = query.lower()

        symptom_keywords = [
            'đau đầu', 'sốt', 'ho', 'khó thở', 'đau bụng', 'buồn nôn', 'nôn',
            'tiêu chảy', 'táo bón', 'đau ngực', 'tim đập nhanh', 'chóng mặt',
            'mệt mỏi', 'khàn tiếng', 'đau họng', 'sưng', 'viêm', 'nhiễm trùng'
        ]

        for title, content in self.context_dict.items():
            content_lower = content.lower()

            if any(symptom in query_lower for symptom in symptom_keywords):
                if any(keyword in content_lower for keyword in query_lower.split()):
                    relevant_docs.append(
                        Document(page_content=content, metadata={"source": title})
                    )
            elif any(keyword in content_lower for keyword in query_lower.split()):
                relevant_docs.append(
                    Document(page_content=content, metadata={"source": title})
                )

        return relevant_docs

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        return self._get_relevant_documents(query)


# =========================
# Medical RAG System
# =========================
class MedicalRAGSystem:
    def __init__(self, use_openai_embeddings=True):
        self.use_openai_embeddings = use_openai_embeddings
        self.vectorstore = None
        self.chunks = None
        self.conversation_chain = None
        self.system_message = "Bạn là một trợ lý y tế AI."

    # ---- Embeddings
    def setup_embeddings(self):
        if self.use_openai_embeddings and os.getenv("OPENAI_API_KEY"):
            print("Using OpenAI embeddings...")
            return OpenAIEmbeddings()
        else:
            print("Using HuggingFace embeddings...")
            return HuggingFaceEmbeddings(
                model_name="sentence-transformers/all-MiniLM-L6-v2"
            )

    # ---- Load documents
    def load_documents(self):
        medical_folders = [
            "medical-knowledge/diseases",
            "medical-knowledge/symptoms",
            "medical-knowledge/departments",
            "medical-knowledge/doctors",
            "medical-knowledge/treatments",
            "medical-knowledge/health-tips",
            "medical-knowledge/services",
        ]
        for folder in medical_folders:
            os.makedirs(folder, exist_ok=True)

        folders = glob.glob("medical-knowledge/*")
        text_loader_kwargs = {"autodetect_encoding": True}
        documents = []

        for folder in folders:
            if os.path.isdir(folder):
                doc_type = os.path.basename(folder)
                loader = DirectoryLoader(
                    folder,
                    glob="**/*.md",
                    loader_cls=TextLoader,
                    loader_kwargs=text_loader_kwargs,
                )
                try:
                    folder_docs = loader.load()
                    for doc in folder_docs:
                        doc.metadata["doc_type"] = doc_type
                        documents.append(doc)
                except Exception as e:
                    print(f"Error loading from {folder}: {e}")

        if not documents:
            documents = self._create_sample_medical_documents()

        print(f"Total medical documents loaded: {len(documents)}")
        return documents

    # ---- Sample docs
    def _create_sample_medical_documents(self):
        sample_docs = [
            Document(
                page_content="""# Cảm cúm (Influenza)

## Triệu chứng:
- Sốt cao đột ngột (38-40°C)
- Đau đầu, đau cơ
- Ho, hắt hơi
- Mệt mỏi, chán ăn

## Nguyên nhân:
- Virus cảm cúm (Influenza A, B, C)
- Lây truyền qua đường hô hấp

## Khoa khám: Khoa Nội Tổng Quát hoặc Khoa Hô Hấp""",
                metadata={"doc_type": "diseases", "source": "cam_cum.md"},
            ),
            Document(
                page_content="""# Viêm họng

## Triệu chứng:
- Đau họng, khó nuốt
- Sưng amidan
- Sốt nhẹ
- Khàn tiếng
- Ho khô

## Nguyên nhân:
- Virus hoặc vi khuẩn
- Kích ứng do khói bụi, lạnh

## Khoa khám: Khoa Tai Mũi Họng""",
                metadata={"doc_type": "diseases", "source": "viem_hong.md"},
            ),
            Document(
                page_content="""# Bác sĩ Nguyễn Văn An
## Chuyên khoa: Nội Tổng Quát
## Kinh nghiệm: 15 năm
## Chứng chỉ: Bác sĩ CKI Nội khoa, Siêu âm Tổng quát
## Thời gian làm việc: Thứ 2-7, sáng và chiều""",
                metadata={"doc_type": "doctors", "source": "bs_nguyen_van_an.md"},
            ),
        ]

        for doc in sample_docs:
            folder_path = f"medical-knowledge/{doc.metadata['doc_type']}"
            os.makedirs(folder_path, exist_ok=True)
            file_path = os.path.join(folder_path, doc.metadata["source"])
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(doc.page_content)

        return sample_docs

    # ---- Split
    def create_chunks(self, documents):
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=800, chunk_overlap=100, separators=["\n\n", "\n", ". ", " ", ""]
        )
        chunks = text_splitter.split_documents(documents)
        print(f"Created {len(chunks)} chunks")
        return chunks

    # ---- Vectorstore
    def create_vectorstore(self, chunks, embeddings):
        if os.path.exists(DB_NAME):
            try:
                Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()
            except Exception:
                pass

        vectorstore = Chroma.from_documents(
            documents=chunks, embedding=embeddings, persist_directory=DB_NAME
        )
        print(f"Vector store created with {vectorstore._collection.count()} documents")
        return vectorstore

    # ---- Load context
    def load_medical_context(self):
        context = {}
        folders = glob.glob("medical-knowledge/*")
        for folder in folders:
            if os.path.isdir(folder):
                for file in glob.glob(f"{folder}/*.md"):
                    try:
                        with open(file, "r", encoding="utf-8") as f:
                            context[os.path.basename(file)] = f.read()
                    except Exception as e:
                        print(f"Error reading {file}: {e}")
        return context

    # ---- Conversation chain
    def setup_conversation_chain(self):
        llm = ChatOpenAI(temperature=0.7, model_name=MODEL)
        memory = ConversationBufferWindowMemory(
            memory_key='chat_history', return_messages=True
        )

        self.context = self.load_medical_context()
        keyword_retriever = MedicalKeywordRetriever(context_dict=self.context)
        vector_retriever = self.vectorstore.as_retriever(search_kwargs={"k": 20})

        hybrid_retriever = EnsembleRetriever(
            retrievers=[keyword_retriever, vector_retriever],
            weights=[0.4, 0.6]
        )

        self.conversation_chain = ConversationalRetrievalChain.from_llm(
            llm=llm, retriever=hybrid_retriever, memory=memory
        )

    # ---- Initialize
    def initialize(self):
        print("Initializing Medical RAG System...")
        embeddings = self.setup_embeddings()
        documents = self.load_documents()
        self.chunks = self.create_chunks(documents)
        self.vectorstore = self.create_vectorstore(self.chunks, embeddings)
        self.setup_conversation_chain()
        print("Medical RAG System initialized successfully!")

    # ---- Chat
    def chat(self, message, history):
        try:
            enhanced_message = f"""
            {self.system_message}

            Câu hỏi của bệnh nhân: {message}

            Hãy phân tích và đưa ra:
            1. Dự đoán bệnh lý (nếu có)
            2. Khoa khám phù hợp
            3. Khuyến cáo
            4. Thông tin bác sĩ (nếu có)
            """
            result = self.conversation_chain.invoke({"question": enhanced_message})
            return result["answer"]
        except Exception as e:
            return f"Lỗi: {str(e)}"

    # ---- Interface
    def launch_interface(self):
        interface = gr.ChatInterface(
            fn=self.chat,
            type="messages",
            title="🏥 Trợ lý Y tế AI",
            description="Dự đoán bệnh, khoa khám, khuyến cáo và thông tin bác sĩ.",
            examples=[
                "Tôi bị sốt cao, đau đầu và ho. Có phải cảm cúm không?",
                "Đau bụng, buồn nôn nên khám khoa nào?",
                "Cho tôi biết thông tin về bác sĩ chuyên khoa tim mạch",
                "Làm thế nào để phòng ngừa cảm cúm?",
            ],
        )
        return interface.launch(inbrowser=True)


# =========================
# Main
# =========================
if __name__ == "__main__":
    medical_system = MedicalRAGSystem(use_openai_embeddings=False)  # HuggingFace
    medical_system.initialize()
    medical_system.launch_interface()


Initializing Medical RAG System...
Using HuggingFace embeddings...
Total medical documents loaded: 66
Created 45 chunks
Vector store created with 45 documents
Medical RAG System initialized successfully!
* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
